In [1]:
from datamodules.bert_datamodule import BertDataModule
import hydra
from models.bert_module import CustomBertModelModule
import pytorch_lightning as pl
from aggregator import Aggregator
from client import Client

/home/aladin/resilient_sfl/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
# Hyperparameters
num_clients = 2
num_rounds = 2
num_epochs = 2

# Module instantiation
model = CustomBertModelModule.from_pretrained("bert-base-uncased", num_labels=2)
datamodule = BertDataModule(glue_dataset="sst2",
                            batch_size=32,
                            num_workers=1,
                            model_type="bert-base-uncased",
                            truncate=100,
                            num_splits=num_clients)

# Data split
datamodule.prepare_data()
datamodule.setup()
train_dls = datamodule.train_dataloader()
val_dls = datamodule.val_dataloader()

# Instantiate clients 
clients = []
for i in range(num_clients):
    client = Client(name=f"client_{i+1}",
                    model=model,
                    trainer=pl.Trainer(default_root_dir="lightning_logs/",
                                       min_epochs=1,
                                       max_epochs=num_epochs,
                                       accelerator="gpu",
                                       devices=1,
                                       limit_train_batches=100,
                                       limit_val_batches=10,
                                       limit_test_batches=10,
                                       check_val_every_n_epoch=1,
                                       deterministic=False),
                    train_data=train_dls[i],
                    val_data=val_dls[i])
    clients.append(client)

# Instantiate aggregator
aggregator = Aggregator(name="sfl_aggregator")

Some weights of the model checkpoint at bert-base-uncased were not used when initializing CustomBertModelModule: ['cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.bias', 'cls.seq_relationship.weight', 'cls.predictions.decoder.weight']
- This IS expected if you are initializing CustomBertModelModule from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing CustomBertModelModule from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of CustomBertModelModule were not initialized from the model checkpoint at bert-base-uncased and are newly

In [11]:
# SFL global round loop
for r in range(num_rounds):
    
    print(f"GLOBAL ROUND : {r+1} of {num_rounds}")

    # Train client models
    for client in clients:
        # Reset trainer to avoid max_epochs boundary
        client.trainer = pl.Trainer(default_root_dir="lightning_logs/",
                                       min_epochs=1,
                                       max_epochs=num_epochs,
                                       accelerator="gpu",
                                       devices=1,
                                       limit_train_batches=100,
                                       limit_val_batches=10,
                                       limit_test_batches=10,
                                       check_val_every_n_epoch=1,
                                       deterministic=False)
        client.trainer.fit(client.model, client.train_data, client.val_data)

    print("ALL CLIENTS TRAINED")

    # Aggregate client models
    attentions = aggregator.accumulate_attentions([client.model for client in clients])
    heads = aggregator.accumulate_heads([client.model for client in clients])
    embeddings = aggregator.accumulate_embeddings([client.model for client in clients])

    aggregated_attentions = aggregator.aggregate(attentions)
    aggregated_heads = aggregator.aggregate(heads)
    aggregated_embeddings = aggregator.aggregate(embeddings)

    # Model update and save
    for client in clients:
        client.update_model(aggregated_attentions)
        client.update_model(aggregated_heads)
        client.update_model(aggregated_embeddings)

    if r == num_rounds - 1:
        print("ALL ROUNDS COMPLETED")

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


GLOBAL ROUND : 1 of 2


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name       | Type              | Params
-------------------------------------------------
0 | bert       | BertModel         | 109 M 
1 | dropout    | Dropout           | 0     
2 | classifier | Linear            | 1.5 K 
3 | embedding  | BertModel         | 109 M 
4 | attention  | CustomBertEncoder | 85.1 M
5 | accuracy   | BinaryAccuracy    | 0     
-------------------------------------------------
304 M     Trainable params
0         Non-trainable params
304 M     Total params
1,216.082 Total estimated model params size (MB)


Epoch 1: 100%|██████████| 4/4 [00:01<00:00,  3.93it/s, loss=0.844, v_num=37, val_loss=0.959, val_acc=0.500, train_loss=0.736, train_acc=0.520]

`Trainer.fit` stopped: `max_epochs=2` reached.


...

Epoch 1: 100%|██████████| 4/4 [00:04<00:00,  1.03s/it, loss=0.734, v_num=39, val_loss=0.710, val_acc=0.500, train_loss=0.724, train_acc=0.600]


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name       | Type              | Params
-------------------------------------------------
0 | bert       | BertModel         | 109 M 
1 | dropout    | Dropout           | 0     
2 | classifier | Linear            | 1.5 K 
3 | embedding  | BertModel         | 109 M 
4 | attention  | CustomBertEncoder | 85.1 M
5 | accuracy   | BinaryAccuracy    | 0     
-------------------------------------------------
304 M     Trainable params
0         Non-trainable params
304 M     Total params
1,216.082 Total estimated model params size (MB)


Epoch 1: 100%|██████████| 4/4 [00:00<00:00,  4.20it/s, loss=0.679, v_num=40, val_loss=0.691, val_acc=0.540, train_loss=0.676, train_acc=0.600]

`Trainer.fit` stopped: `max_epochs=2` reached.


Epoch 1: 100%|██████████| 4/4 [00:04<00:00,  1.08s/it, loss=0.679, v_num=40, val_loss=0.691, val_acc=0.540, train_loss=0.676, train_acc=0.600]
ALL CLIENTS TRAINED
ALL ROUNDS COMPLETED


In [12]:
# Aggregate client models
attentions = aggregator.accumulate_attentions([client.model for client in clients])
heads = aggregator.accumulate_heads([client.model for client in clients])
embeddings = aggregator.accumulate_embeddings([client.model for client in clients])

aggregated_attentions = aggregator.aggregate(attentions)
aggregated_heads = aggregator.aggregate(heads)
aggregated_embeddings = aggregator.aggregate(embeddings)

In [23]:
import torch
a = attentions[0]
b = attentions[1]

for key in a.keys():
    if not torch.allclose(a[key], b[key]):
        print(key)
    else:
        pass


# print(a['bert.encoder.layer.0.attention.self.query.weight'] == b['bert.encoder.layer.0.attention.self.query.weight'])